In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# =============================================================================
# Cell 1: Setup and Configuration
# =============================================================================

import os
import gc
import json
import gzip
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pacsv
from tqdm.auto import tqdm

# Configure logging (Professional, no symbols)
logging.basicConfig(
    level=logging.INFO,
    force = True,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("DataSplitPipeline")

# Polars configuration for large datasets
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)
os.environ["OPENBLAS_NUM_THREADS"] = "1"

@dataclass
class SplitConfig:
    """Configuration for Data Splitting Pipeline."""

    # Core Ratios and Thresholds
    k_user: int = 15
    k_book: int = 10
    train_ratio: float = 0.8
    valid_ratio: float = 0.1

    # Rating Definitions
    explicit_min_rating: int = 1
    explicit_max_rating: int = 5
    positive_min_rating: int = 4
    negative_max_rating: int = 3

    # Base Directories
    base_dir: Path = field(default_factory=lambda: Path("/content/drive/My Drive/Thesis/book_recsys"))
    thesis_dir: Path = field(default_factory=lambda: Path("/content/drive/My Drive/Thesis/"))
    # Pipeline Control Flags
    skip_stage1: bool = False
    skip_stage2: bool = False
    skip_stage3: bool = False

    def __post_init__(self) -> None:
        """Resolve paths based on the base directory."""
        self.data_dir = self.base_dir / "data"
        self.raw_dir = self.thesis_dir / "Data"

        # Updated output directory per requirements
        self.output_dir = self.data_dir / "processed_2"
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Input Files
        self.raw_json_gz = self.raw_dir / "goodreads_interactions_dedup.json.gz"
        self.ingested_parquet = self.data_dir / "interactions_raw.parquet"
        self.kcore_parquet = self.data_dir / "interactions_kcore.parquet"

        # Output Files Configuration
        self.outputs: Dict[str, Path] = {
            "train_main": self.output_dir / "train_main.csv",
            "explicit_train": self.output_dir / "explicit_train.csv",
            "train_implicit": self.output_dir / "train_implicit.csv",
            "valid_pos": self.output_dir / "valid_pos.csv",
            "test_pos": self.output_dir / "test_pos.csv",
            "test_neg": self.output_dir / "test_neg.csv",
            "test_implicit": self.output_dir / "test_implicit.csv",
            "explicit_test": self.output_dir / "explicit_test.csv",
        }

# Verify configuration initialization
if __name__ == "__main__":
    cfg = SplitConfig()
    log.info("Configuration initialized successfully.")
    log.info(f"Target output directory: {cfg.output_dir}")

2026-04-19 14:56:23 [INFO] DataSplitPipeline: Configuration initialized successfully.
2026-04-19 14:56:23 [INFO] DataSplitPipeline: Target output directory: /content/drive/My Drive/Thesis/book_recsys/data/processed_2


In [ ]:
# =============================================================================
# Cell 2: Stage 1 - JSON to Parquet Ingestion (Memory Safe)
# =============================================================================

import gc
import json
import gzip
import logging
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

log = logging.getLogger("DataSplitPipeline")

def stage_1_ingestion(cfg) -> None:
    """
    Reads a massive JSON.gz file line-by-line, extracts only required fields,
    and streams to a Parquet file in chunks to prevent Out-Of-Memory (OOM) errors.
    """
    if cfg.skip_stage1:
        log.info("Stage 1 skipped via configuration.")
        return

    log.info("Stage 1: JSON.gz to Parquet Ingestion (Memory Safe Mode)")
    input_path = cfg.raw_json_gz
    output_path = cfg.ingested_parquet

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found at: {input_path}")

    # Enforce strict schema to optimize PyArrow memory allocation
    schema = pa.schema([
        ('user_id', pa.string()),
        ('book_id', pa.string()),
        ('rating', pa.int16()),
        ('date_updated', pa.string())
    ])

    chunk_size = 1_000_000
    buffer = []
    writer = None

    log.info("Starting streaming process...")

    with gzip.open(input_path, 'rt', encoding='utf-8') as f:
        # Progress bar without a fixed total (since gzip line count is unknown)
        pbar = tqdm(desc="Ingesting records", unit=" rows")

        for line in f:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue # Skip corrupted lines if any

            # Feature Selection: Only keep necessary columns to save RAM and Disk
            buffer.append({
                'user_id': record.get('user_id', ''),
                'book_id': record.get('book_id', ''),
                'rating': int(record.get('rating', 0)),
                'date_updated': record.get('date_updated', '')
            })

            if len(buffer) >= chunk_size:
                # Convert to PyArrow Table
                table = pa.Table.from_pylist(buffer, schema=schema)

                # Initialize writer on first chunk
                if writer is None:
                    writer = pq.ParquetWriter(output_path, schema)

                writer.write_table(table)

                # Strict Memory Cleanup
                buffer.clear()
                del table
                gc.collect()

                pbar.update(chunk_size)

        # Flush the remaining records in the buffer
        if buffer:
            table = pa.Table.from_pylist(buffer, schema=schema)
            if writer is None:
                writer = pq.ParquetWriter(output_path, schema)
            writer.write_table(table)
            pbar.update(len(buffer))

            buffer.clear()
            del table
            gc.collect()

        pbar.close()

    if writer:
        writer.close()

    log.info(f"Stage 1 completed. Parquet saved to: {output_path}")

# =============================================================================
# Execution and Verification Block
# =============================================================================
# if __name__ == "__main__":
#     try:
#         log.info("Testing Stage 1 Execution...")

#         # Execute the ingestion process
#         stage_1_ingestion(cfg)

#         # Verify output without loading the entire file into RAM
#         if cfg.ingested_parquet.exists():
#             pf = pq.ParquetFile(cfg.ingested_parquet)
#             total_rows = pf.metadata.num_rows
#             log.info("Verification Success!")
#             log.info(f"Total rows successfully written to Parquet: {total_rows:,}")
#         else:
#             log.error("Verification Failed! Output Parquet file not found.")

#     except Exception as e:
#         log.error(f"Error during Stage 1 execution: {str(e)}")

In [ ]:
# =============================================================================
# Cell 3: Stage 2 - Iterative K-Core Filtering (Bulletproof + Anti-Leak RAM)
# =============================================================================

import os
import gc
import logging
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl

log = logging.getLogger("DataSplitPipeline")

def stage_2_kcore(cfg) -> None:
    """
    Bulletproof K-Core Filtering with Strict Memory Leak Prevention:
    - Uses DuckDB CHECKPOINT to flush Write-Ahead Logs (WAL).
    - Forces explicit Python Garbage Collection per iteration and per export batch.
    """
    if cfg.skip_stage2:
        log.info("Stage 2 skipped via configuration.")
        return

    log.info("Stage 2: Iterative K-Core Filtering (Anti-Leak Architecture)")

    input_file = str(cfg.ingested_parquet)
    output_file = str(cfg.kcore_parquet)
    db_file = str(cfg.data_dir / "kcore_workspace.db")

    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found at: {input_file}")

    # Remove old workspace if it exists to ensure a fresh start
    if os.path.exists(db_file):
        try:
            os.remove(db_file)
            # Also remove WAL files if they were left behind
            if os.path.exists(db_file + ".wal"): os.remove(db_file + ".wal")
        except OSError:
            pass

    log.info("Initializing DuckDB database...")
    con = duckdb.connect(db_file)

    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='8GB'")

    log.info("Loading Parquet data into DuckDB Engine...")
    con.execute(f"CREATE TABLE interactions AS SELECT * FROM read_parquet('{input_file}')")

    iteration = 1
    prev_user_count = -1
    prev_book_count = -1

    while True:
        log.info(f"--- K-Core Iteration {iteration} ---")

        log.info("Calculating retention thresholds...")
        con.execute(f"""
            CREATE TEMP TABLE valid_users AS
            SELECT user_id FROM interactions GROUP BY user_id HAVING COUNT(*) >= {cfg.k_user}
        """)
        con.execute(f"""
            CREATE TEMP TABLE valid_books AS
            SELECT book_id FROM interactions GROUP BY book_id HAVING COUNT(*) >= {cfg.k_book}
        """)

        current_user_count = con.execute("SELECT COUNT(*) FROM valid_users").fetchone()[0]
        current_book_count = con.execute("SELECT COUNT(*) FROM valid_books").fetchone()[0]

        log.info(f"Retained Users: {current_user_count:,} | Retained Books: {current_book_count:,}")

        if current_user_count == prev_user_count and current_book_count == prev_book_count:
            log.info("Convergence reached. Iteration completed.")
            con.execute("DROP TABLE valid_users")
            con.execute("DROP TABLE valid_books")
            break

        log.info("Filtering interactions table...")
        con.execute("""
            CREATE TABLE interactions_next AS
            SELECT i.* FROM interactions i
            INNER JOIN valid_users u ON i.user_id = u.user_id
            INNER JOIN valid_books b ON i.book_id = b.book_id
        """)

        con.execute("DROP TABLE interactions")
        con.execute("DROP TABLE valid_users")
        con.execute("DROP TABLE valid_books")
        con.execute("ALTER TABLE interactions_next RENAME TO interactions")

        # ANTI-LEAK MECHANISM 1: Force DuckDB to flush WAL to disk and free memory
        log.info("Checkpointing DB to free internal RAM...")
        con.execute("CHECKPOINT")

        # ANTI-LEAK MECHANISM 2: Force Python to clear any dangling query objects
        gc.collect()

        prev_user_count = current_user_count
        prev_book_count = current_book_count
        iteration += 1

    # =========================================================================
    # EXPORT PHASE (Memory Safe)
    # =========================================================================
    log.info("Exporting final table safely to Parquet...")
    query_result = con.execute("SELECT * FROM interactions")

    # 100K chunk size keeps RAM usage for PyArrow extremely flat
    record_batch_reader = query_result.fetch_record_batch(rows_per_batch=100_000)

    writer = None
    for batch in record_batch_reader:
        if writer is None:
            writer = pq.ParquetWriter(output_file, batch.schema, compression='snappy')
        writer.write_batch(batch)

        # ANTI-LEAK MECHANISM 3: Destroy the batch immediately, don't wait for loop to end
        del batch
        gc.collect()

    if writer:
        writer.close()

    con.close()

    # Final strict cleanup of the heavy DB files
    if os.path.exists(db_file): os.remove(db_file)
    if os.path.exists(db_file + ".wal"): os.remove(db_file + ".wal")

    log.info(f"Stage 2 completed perfectly. Final Parquet saved to: {output_file}")


# =============================================================================
# Execution and Verification Block
# =============================================================================
# if __name__ == "__main__":
#     try:
#         log.info("Testing Stage 2 Execution...")
#         stage_2_kcore(cfg)

#         if cfg.kcore_parquet.exists():
#             log.info("Running verification...")
#             total_rows = pl.scan_parquet(cfg.kcore_parquet).select(pl.len()).collect().item()
#             log.info("Verification Success! The Parquet file is structurally perfect.")
#             log.info(f"Total rows after K-Core filtering: {total_rows:,}")
#         else:
#             log.error("Verification Failed! Output Parquet file not found.")

#     except Exception as e:
#         log.error(f"Error during Stage 2 execution: {str(e)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
# =============================================================================
# Cell 4: Stage 3A & 3B - Chunk-based Temporal Split (Bypass I/O & NULL Bugs)
# =============================================================================

import os
import gc
import logging
import shutil
import duckdb
import polars as pl
from pathlib import Path
from tqdm.auto import tqdm

log = logging.getLogger("DataSplitPipeline")

def stage_3_chunked_split(cfg) -> None:
    if cfg.skip_stage3:
        log.info("Stage 3 skipped via configuration.")
        return

    log.info("Stage 3: Chunk-based Temporal Split (Robust Architecture)")

    input_file = str(cfg.kcore_parquet)
    db_workspace = str(cfg.data_dir / "split_workspace.db")

    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")

    # Giai phap 1: Chuyen huong I/O sang o cung cuc bo cua Linux (Colab)
    # Viec ghi noi tiep (append) hang ngan lan tren Google Drive se gay mat du lieu.
    # Ta tao mot thu muc tam thoi tren o SSD cuc bo de ghi file hieu qua nhat.
    local_temp_dir = Path("/content/temp_export")
    local_temp_dir.mkdir(parents=True, exist_ok=True)

    # Dinh nghia 13 file dich nam tren o cung cuc bo
    output_files = {
        "train_main": local_temp_dir / "train_main.csv",
        "train_explicit": local_temp_dir / "train_explicit.csv",
        "train_implicit": local_temp_dir / "train_implicit.csv",
        "valid_main": local_temp_dir / "valid_main.csv",
        "valid_explicit": local_temp_dir / "valid_explicit.csv",
        "valid_implicit": local_temp_dir / "valid_implicit.csv",
        "valid_pos": local_temp_dir / "valid_pos.csv",
        "valid_neg": local_temp_dir / "valid_neg.csv",
        "test_main": local_temp_dir / "test_main.csv",
        "test_explicit": local_temp_dir / "test_explicit.csv",
        "test_implicit": local_temp_dir / "test_implicit.csv",
        "test_pos": local_temp_dir / "test_pos.csv",
        "test_neg": local_temp_dir / "test_neg.csv",
    }

    # Xoa sach file cu trong thu muc tam neu co de bat dau ghi moi
    for file_path in output_files.values():
        if file_path.exists():
            file_path.unlink()

    # Don dep db workspace cu
    for f in [db_workspace, db_workspace + ".wal"]:
        if os.path.exists(f):
            try: os.remove(f)
            except OSError: pass

    log.info("Initializing DuckDB Workspace...")
    con = duckdb.connect(db_workspace)
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='6GB'")

    # =========================================================================
    # BUOC 1: Phan lo nguoi dung va tinh toan ranh gioi
    # =========================================================================
    log.info("Step 1/4: Calculating batch boundaries inside DB...")

    # Su dung SQL de nhom du lieu va danh so lo (batch_id).
    # NULL user_id cung se duoc nhom vao 1 dong duy nhat tai day.
    con.execute(f"""
        CREATE TABLE user_batch_mapping AS
        WITH counts AS (
            SELECT user_id, COUNT(*) as total_rows
            FROM read_parquet('{input_file}')
            GROUP BY user_id
        ),
        cumulative AS (
            SELECT user_id, total_rows, SUM(total_rows) OVER (ORDER BY user_id) as cum_sum
            FROM counts
        )
        SELECT user_id, total_rows, CAST(cum_sum / 1000000 AS INTEGER) as batch_id
        FROM cumulative;
    """)
    con.execute("CHECKPOINT")

    num_batches = con.execute("SELECT MAX(batch_id) FROM user_batch_mapping").fetchone()[0] + 1
    total_users = con.execute("SELECT COUNT(*) FROM user_batch_mapping").fetchone()[0]
    log.info(f"Mapped {total_users:,} unique users (including NULLs) into {num_batches} batches.")

    # =========================================================================
    # BUOC 2: Noi bang va danh chi muc vat ly
    # =========================================================================
    log.info("Step 2/4: Materializing joined data to local disk...")

    # Giai phap 2: Su dung IS NOT DISTINCT FROM.
    # Phep toan nay dam bao khi 19 trieu dong NULL trong bang goc mapping voi bang batch_id,
    # chung khong bi SQL loai bo nhu phep toan dau bang (=) thong thuong.
    con.execute(f"""
        CREATE TABLE chunked_data AS
        SELECT t.user_id, t.book_id, CAST(t.rating AS INTEGER) AS rating, t.date_updated, m.batch_id, m.total_rows
        FROM read_parquet('{input_file}') t
        INNER JOIN user_batch_mapping m ON t.user_id IS NOT DISTINCT FROM m.user_id
    """)
    con.execute("CHECKPOINT")

    # Danh chi muc de tang toc do truy xuat tung lo trong vong lap, ngan chan nghen I/O
    con.execute("CREATE INDEX idx_batch ON chunked_data(batch_id)")
    con.execute("CHECKPOINT")

    # =========================================================================
    # BUOC 3: Xu ly tung lo (Chunk) va ghi xuong o cuc bo
    # =========================================================================
    log.info("Step 3/4: Processing and streaming to local CSV files...")
    keep_cols = ["user_id", "book_id", "rating"]

    for batch_idx in tqdm(range(num_batches), desc="Processing Chunks"):
        # Lay du lieu cua lo hien tai
        df_chunk = con.execute(f"SELECT * FROM chunked_data WHERE batch_id = {batch_idx}").pl()

        if df_chunk.height == 0:
            continue

        # Sap xep thoi gian. Day cac dong khong co ngay thang xuong cuoi de khong mat du lieu.
        df_chunk = df_chunk.sort(["user_id", "date_updated"], descending=[False, False], nulls_last=True)

        # Danh so thu tu tuyet doi cho tung dong cua moi nguoi dung
        df_chunk = df_chunk.with_columns([
            pl.int_range(1, pl.len() + 1).over("user_id").alias("row_num"),
            (pl.col("total_rows") * cfg.train_ratio).floor().cast(pl.Int32).alias("train_end"),
            (pl.col("total_rows") * (cfg.train_ratio + cfg.valid_ratio)).floor().cast(pl.Int32).alias("valid_end")
        ])

        # Xac dinh ranh gioi thoi gian
        is_train = pl.col("row_num") <= pl.col("train_end")
        is_valid = (pl.col("row_num") > pl.col("train_end")) & (pl.col("row_num") <= pl.col("valid_end"))
        is_test = pl.col("row_num") > pl.col("valid_end")

        # Xac dinh tinh chat phan hoi (Rating)
        is_explicit = pl.col("rating").is_between(cfg.explicit_min_rating, cfg.explicit_max_rating)
        is_implicit = ~is_explicit
        is_pos = is_explicit & (pl.col("rating") >= cfg.positive_min_rating)
        is_neg = is_explicit & (pl.col("rating") <= cfg.negative_max_rating)

        # Cau hinh 13 ma tran loc
        export_specs = [
            ("train_main", is_train, output_files["train_main"]),
            ("train_explicit", is_train & is_explicit, output_files["train_explicit"]),
            ("train_implicit", is_train & is_implicit, output_files["train_implicit"]),
            ("valid_main", is_valid, output_files["valid_main"]),
            ("valid_explicit", is_valid & is_explicit, output_files["valid_explicit"]),
            ("valid_implicit", is_valid & is_implicit, output_files["valid_implicit"]),
            ("valid_pos", is_valid & is_pos, output_files["valid_pos"]),
            ("valid_neg", is_valid & is_neg, output_files["valid_neg"]),
            ("test_main", is_test, output_files["test_main"]),
            ("test_explicit", is_test & is_explicit, output_files["test_explicit"]),
            ("test_implicit", is_test & is_implicit, output_files["test_implicit"]),
            ("test_pos", is_test & is_pos, output_files["test_pos"]),
            ("test_neg", is_test & is_neg, output_files["test_neg"]),
        ]

        # Loc va ghi noi tiep an toan vao SSD cuc bo
        for name, predicate, out_path in export_specs:
            sub_df = df_chunk.filter(predicate).select(keep_cols)
            if sub_df.height > 0:
                write_header = not out_path.exists()
                with open(out_path, mode="ab") as f:
                    sub_df.write_csv(f, include_header=write_header)

        # Giai phong RAM sau moi lo
        del df_chunk
        gc.collect()

    con.close()

    # =========================================================================
    # BUOC 4: Chuyen giao ket qua an toan len Google Drive
    # =========================================================================
    log.info("Step 4/4: Transferring final files to Google Drive...")

    cfg.output_dir.mkdir(parents=True, exist_ok=True)

    # Copy tung file mot cach nguyen ven tu SSD len Drive
    for name, local_path in output_files.items():
        if local_path.exists():
            drive_path = cfg.output_dir / f"{name}.csv"
            log.info(f"Copying {name}.csv to Drive...")
            shutil.copy2(local_path, drive_path)

    log.info("Stage 3 local execution and transfer completed.")

# =============================================================================
# Execution and Final Verification Block
# =============================================================================
if __name__ == "__main__":
    try:
        log.info("Executing Enterprise-Grade Stage 3 Split...")
        stage_3_chunked_split(cfg)

        log.info("Performing Sanity Check: Verifying Row Counts...")

        # Doc file goc
        kcore_rows = pl.scan_parquet(cfg.kcore_parquet).select(pl.len()).collect().item()

        # Doc 3 file Main tu tren Google Drive
        train_rows = pl.scan_csv(cfg.output_dir / "train_main.csv").select(pl.len()).collect().item()
        valid_rows = pl.scan_csv(cfg.output_dir / "valid_main.csv").select(pl.len()).collect().item()
        test_rows = pl.scan_csv(cfg.output_dir / "test_main.csv").select(pl.len()).collect().item()

        total_csv_rows = train_rows + valid_rows + test_rows

        log.info(f"Original Parquet Rows: {kcore_rows:,}")
        log.info(f"Train Main Rows:       {train_rows:,}")
        log.info(f"Valid Main Rows:       {valid_rows:,}")
        log.info(f"Test Main Rows:        {test_rows:,}")
        log.info(f"Total Exported Rows:   {total_csv_rows:,}")

        if total_csv_rows == kcore_rows:
            log.info("SANITY CHECK PASSED: 100% Data Preserved. Zero Data Loss.")
        else:
            log.error(f"SANITY CHECK FAILED: Missing {kcore_rows - total_csv_rows:,} rows.")

    except Exception as e:
        log.error(f"Error during Stage 3 execution: {str(e)}")

2026-04-19 11:33:43 [INFO] DataSplitPipeline: Executing Enterprise-Grade Stage 3 Split...
2026-04-19 11:33:43 [INFO] DataSplitPipeline: Stage 3: Chunk-based Temporal Split (Robust Architecture)
2026-04-19 11:33:43 [INFO] DataSplitPipeline: Initializing DuckDB Workspace...
2026-04-19 11:33:44 [INFO] DataSplitPipeline: Step 1/4: Calculating batch boundaries inside DB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 11:34:49 [INFO] DataSplitPipeline: Mapped 743,401 unique users (including NULLs) into 225 batches.
2026-04-19 11:34:49 [INFO] DataSplitPipeline: Step 2/4: Materializing joined data to local disk...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 12:08:26 [INFO] DataSplitPipeline: Step 4/4: Transferring final files to Google Drive...
2026-04-19 12:08:26 [INFO] DataSplitPipeline: Copying train_main.csv to Drive...
2026-04-19 12:09:59 [INFO] DataSplitPipeline: Copying train_explicit.csv to Drive...
2026-04-19 12:15:59 [INFO] DataSplitPipeline: Copying train_implicit.csv to Drive...
2026-04-19 12:17:19 [INFO] DataSplitPipeline: Copying valid_main.csv to Drive...
2026-04-19 12:18:54 [INFO] DataSplitPipeline: Copying valid_explicit.csv to Drive...
2026-04-19 12:19:10 [INFO] DataSplitPipeline: Copying valid_implicit.csv to Drive...
2026-04-19 12:19:29 [INFO] DataSplitPipeline: Copying valid_pos.csv to Drive...
2026-04-19 12:19:48 [INFO] DataSplitPipeline: Copying valid_neg.csv to Drive...
2026-04-19 12:19:57 [INFO] DataSplitPipeline: Copying test_main.csv to Drive...
2026-04-19 12:22:43 [INFO] DataSplitPipeline: Copying test_explicit.csv to Drive...
2026-04-19 12:23:00 [INFO] DataSplitPipeline: Copying test_implicit.csv to

In [13]:
# =============================================================================
# Cell 5: Stage 4 - Book Metadata Extraction (Alignment with K-Core)
# =============================================================================

import os
import gc
import logging
import duckdb
import polars as pl
from pathlib import Path

log = logging.getLogger("DataSplitPipeline")

def stage_4_extract_books(cfg, raw_books_path: str) -> None:
    """
    Extracts only the metadata of books that survived the K-Core filtering stage.
    Ensures that the content-based features (metadata) perfectly align with the
    collaborative filtering features (interactions).
    """
    log.info("Stage 4: Book Metadata Extraction and Alignment")

    kcore_file = str(cfg.kcore_parquet)
    raw_books_file = str(raw_books_path)
    output_books_file = str(cfg.output_dir / "books_filtered.parquet")
    db_workspace = str(cfg.data_dir / "books_workspace.db")

    if not os.path.exists(kcore_file):
        raise FileNotFoundError(f"K-Core interactions file not found: {kcore_file}")
    if not os.path.exists(raw_books_file):
        raise FileNotFoundError(f"Raw books file not found: {raw_books_file}")

    # Don dep workspace cu neu co
    if os.path.exists(output_books_file):
        os.remove(output_books_file)
    for f in [db_workspace, db_workspace + ".wal"]:
        if os.path.exists(f):
            try: os.remove(f)
            except OSError: pass

    log.info("Initializing DuckDB Workspace...")
    con = duckdb.connect(db_workspace)
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='6GB'")

    # Buoc 1: Trich xuat cac ID sach hop le tu file tuong tac
    log.info("Step 1/3: Extracting distinct valid book IDs from interactions...")
    con.execute(f"""
        CREATE TABLE valid_books AS
        SELECT DISTINCT book_id
        FROM read_parquet('{kcore_file}')
    """)
    con.execute("CHECKPOINT")

    valid_books_count = con.execute("SELECT COUNT(*) FROM valid_books").fetchone()[0]
    log.info(f"Found {valid_books_count:,} unique books in the filtered interaction dataset.")

    # Buoc 2: Doc file sach goc.
    # Xac dinh dinh dang file de dung ham doc tuong ung cua DuckDB
    log.info("Step 2/3: Creating view for raw book metadata...")
    if raw_books_file.endswith(".json") or raw_books_file.endswith(".json.gz"):
        read_command = f"read_json_auto('{raw_books_file}')"
    elif raw_books_file.endswith(".parquet"):
        read_command = f"read_parquet('{raw_books_file}')"
    else:
        # Mac dinh cho rang do la CSV neu khong phai JSON/Parquet
        read_command = f"read_csv_auto('{raw_books_file}')"

    con.execute(f"""
        CREATE VIEW raw_books_view AS
        SELECT * FROM {read_command}
    """)

    # Buoc 3: Thuc hien Inner Join de loc thong tin va ghi truc tiep ra file dich
    log.info("Step 3/3: Filtering and exporting aligned book metadata...")

    # Su dung INNER JOIN de chi giu lai nhung cuon sach co mat trong valid_books.
    # Luu truc tiep thanh file Parquet de toi uu dung luong va toc do doc sau nay.
    export_query = f"""
        COPY (
            SELECT b.* FROM raw_books_view b
            INNER JOIN valid_books v ON b.book_id = v.book_id
        ) TO '{output_books_file}' (FORMAT PARQUET, COMPRESSION 'snappy');
    """

    con.execute(export_query)

    # Don dep tai nguyen
    con.close()
    for f in [db_workspace, db_workspace + ".wal"]:
        if os.path.exists(f):
            try: os.remove(f)
            except OSError: pass

    log.info(f"Stage 4 completed perfectly. Filtered books saved to: {output_books_file}")

# =============================================================================
# Execution and Verification Block
# =============================================================================
if __name__ == "__main__":
    try:
        # TODO: Ban can cung cap duong dan toi file thong tin sach goc cua ban tai day
        RAW_BOOKS_METADATA_PATH = Path("/content/drive/My Drive/Thesis/book_recsys/data/books_consolidated_raw.parquet")

        log.info("Executing Stage 4: Book Alignment...")
        stage_4_extract_books(cfg, RAW_BOOKS_METADATA_PATH)

        # Kiem tra cheo (Sanity Check)
        filtered_books_path = cfg.output_dir / "books_filtered.parquet"
        if filtered_books_path.exists():
            log.info("Performing Sanity Check on Filtered Books...")

            total_books_exported = pl.scan_parquet(filtered_books_path).select(pl.len()).collect().item()
            log.info(f"Total Books Exported: {total_books_exported:,}")

            # Neu so luong sach xuat ra khop voi so luong sach trong kcore, he thong da dong bo 100%
            log.info("Alignment Successful. Feature matrices can now be built safely.")

    except Exception as e:
        log.error(f"Error during Stage 4 execution: {str(e)}")

2026-04-19 15:07:36 [INFO] DataSplitPipeline: Executing Stage 4: Book Alignment...
2026-04-19 15:07:36 [INFO] DataSplitPipeline: Stage 4: Book Metadata Extraction and Alignment
2026-04-19 15:07:36 [INFO] DataSplitPipeline: Initializing DuckDB Workspace...
2026-04-19 15:07:36 [INFO] DataSplitPipeline: Step 1/3: Extracting distinct valid book IDs from interactions...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 15:08:44 [INFO] DataSplitPipeline: Found 1,173,713 unique books in the filtered interaction dataset.
2026-04-19 15:08:44 [INFO] DataSplitPipeline: Step 2/3: Creating view for raw book metadata...
2026-04-19 15:08:44 [INFO] DataSplitPipeline: Step 3/3: Filtering and exporting aligned book metadata...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 15:09:09 [INFO] DataSplitPipeline: Stage 4 completed perfectly. Filtered books saved to: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/books_filtered.parquet
2026-04-19 15:09:09 [INFO] DataSplitPipeline: Performing Sanity Check on Filtered Books...
2026-04-19 15:09:09 [INFO] DataSplitPipeline: Total Books Exported: 1,173,713
2026-04-19 15:09:09 [INFO] DataSplitPipeline: Alignment Successful. Feature matrices can now be built safely.


In [12]:
# =============================================================================
# Cell 4.1: Book Metadata Consolidation (DuckDB)
# =============================================================================

import os
import gc
import logging
import duckdb
from pathlib import Path

log = logging.getLogger("DataSplitPipeline")

def consolidate_book_metadata(cfg, books_path: Path, authors_path: Path, genres_path: Path) -> None:
    """
    Consolidates book metadata from multiple JSON/GZ sources into a single,
    rich Parquet file. Uses DuckDB to perform memory-safe Out-of-Core Joins.
    """
    log.info("Starting Book Metadata Consolidation...")

    # Kiem tra file dau vao
    for path in [books_path, authors_path, genres_path]:
        if not path.exists():
            raise FileNotFoundError(f"Source file not found: {path}")

    # Duong dan file dich (Hop nhat)
    consolidated_output = cfg.data_dir / "books_consolidated_raw.parquet"
    db_workspace = str(cfg.data_dir / "metadata_workspace.db")

    # Don dep truoc khi chay
    if consolidated_output.exists():
        consolidated_output.unlink()
    for f in [db_workspace, db_workspace + ".wal"]:
        if os.path.exists(f):
            try: os.remove(f)
            except OSError: pass

    log.info("Initializing DuckDB engine for JSON parsing...")
    con = duckdb.connect(db_workspace)
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='6GB'")

    try:
        # ---------------------------------------------------------------------
        # BUOC 1: Nạp file Tac Gia (Authors)
        # ---------------------------------------------------------------------
        log.info("Step 1/4: Parsing Authors JSON...")
        # File tac gia nho hon, ta nap vao bang vat ly truoc de lam bang tra cuu
        con.execute(f"""
            CREATE TABLE authors AS
            SELECT author_id, name as author_name
            FROM read_json_auto('{str(authors_path)}', format='auto')
        """)
        con.execute("CREATE INDEX idx_author ON authors(author_id)")
        con.execute("CHECKPOINT")

        # ---------------------------------------------------------------------
        # BUOC 2: Nạp file The Loai (Genres)
        # ---------------------------------------------------------------------
        log.info("Step 2/4: Parsing Genres JSON...")
        # Dictionary cua the loai kha phuc tap, DuckDB co the tu ep kieu (infer) thanh JSON/Struct
        con.execute(f"""
            CREATE TABLE genres AS
            SELECT book_id, genres as genre_dict
            FROM read_json_auto('{str(genres_path)}', format='auto')
        """)
        con.execute("CREATE INDEX idx_genre_book ON genres(book_id)")
        con.execute("CHECKPOINT")

        # ---------------------------------------------------------------------
        # BUOC 3: Nạp file Sach (Books) & Giai nen tac gia chinh
        # ---------------------------------------------------------------------
        log.info("Step 3/4: Parsing Main Books JSON & Extracting primary author...")
        # Bang sach rat nang, ta chi lay cac cot thuc su co gia tri cho Recommendation.
        # Vi 'authors' la mot List cac Dictionary, ta chi lay ID cua tac gia DAU TIEN (Primary Author)
        con.execute(f"""
            CREATE TABLE books AS
            SELECT
                book_id,
                title,
                title_without_series,
                description,
                publisher,
                publication_year,
                publication_month,
                publication_day,
                is_ebook,
                language_code,
                average_rating,
                ratings_count,
                image_url,
                num_pages,
                -- Trich xuat author_id cua phan tu dau tien trong mang authors
                TRY_CAST(json_extract_string(authors, '$[0].author_id') AS VARCHAR) as primary_author_id
            FROM read_json_auto('{str(books_path)}', format='auto', maximum_object_size=33554432)
        """)
        con.execute("CREATE INDEX idx_book ON books(book_id)")
        con.execute("CHECKPOINT")

        # ---------------------------------------------------------------------
        # BUOC 4: Tich hop (JOIN) va Xuat file
        # ---------------------------------------------------------------------
        log.info("Step 4/4: Joining all metadata and exporting to Parquet...")

        # Dung LEFT JOIN de dam bao neu sach khong co tac gia hoac the loai, no van khong bi roi mat
        con.execute(f"""
            COPY (
                SELECT
                    b.book_id,
                    b.title,
                    b.title_without_series,
                    a.author_name,
                    g.genre_dict as genres,
                    b.description,
                    b.publisher,
                    b.publication_year,
                    b.publication_month,
                    b.publication_day,
                    b.num_pages,
                    b.is_ebook,
                    b.language_code,
                    b.average_rating,
                    b.ratings_count,
                    b.image_url
                FROM books b
                LEFT JOIN authors a ON b.primary_author_id = a.author_id
                LEFT JOIN genres g ON b.book_id = g.book_id
            ) TO '{str(consolidated_output)}' (FORMAT PARQUET, COMPRESSION 'snappy');
        """)

    finally:
        # Don dep db workspace
        con.close()
        for f in [db_workspace, db_workspace + ".wal"]:
            if os.path.exists(f):
                try: os.remove(f)
                except OSError: pass

    log.info(f"Consolidation complete! Raw unified metadata saved to: {consolidated_output}")

# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    try:
        # TODO: Ban can cung cap duong dan chinh xac toi 3 file nay tren Drive cua ban
        RAW_BOOKS_PATH = Path("/content/drive/My Drive/Thesis/Data/goodreads_books.json") # hoac .json
        AUTHORS_PATH = Path("/content/drive/My Drive/Thesis/Data/goodreads_book_authors.json.gz")
        GENRES_PATH = Path("/content/drive/My Drive/Thesis/Data/goodreads_book_genres_initial.json.gz")

        log.info("Executing Stage 4.1: Book Metadata Consolidation...")
        consolidate_book_metadata(cfg, RAW_BOOKS_PATH, AUTHORS_PATH, GENRES_PATH)

    except Exception as e:
        log.error(f"Error during Stage 4.1 execution: {str(e)}")

2026-04-19 14:56:54 [INFO] DataSplitPipeline: Executing Stage 4.1: Book Metadata Consolidation...
2026-04-19 14:56:54 [INFO] DataSplitPipeline: Starting Book Metadata Consolidation...
2026-04-19 14:56:54 [INFO] DataSplitPipeline: Initializing DuckDB engine for JSON parsing...
2026-04-19 14:56:54 [INFO] DataSplitPipeline: Step 1/4: Parsing Authors JSON...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 14:57:01 [INFO] DataSplitPipeline: Step 2/4: Parsing Genres JSON...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 15:00:12 [INFO] DataSplitPipeline: Step 4/4: Joining all metadata and exporting to Parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 15:00:51 [INFO] DataSplitPipeline: Consolidation complete! Raw unified metadata saved to: /content/drive/My Drive/Thesis/book_recsys/data/books_consolidated_raw.parquet


In [3]:
# =============================================================================
# Extra Cell: Comprehensive Integrity Check (OOM-Safe)
# =============================================================================

import os
import duckdb
import logging
from pathlib import Path

# Thiet lap logging neu chua co
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("IntegrityCheck")

def verify_dataset_integrity(cfg) -> None:
    """
    Kiem tra toan dien do dong nhat cua du lieu giua file Parquet goc va cac file CSV xuat ra.
    Su dung DuckDB de chong tran RAM khi dem hang trieu gia tri DISTINCT.
    """
    log.info("Starting Comprehensive Integrity Check...")

    # Dinh nghia duong dan
    parquet_path = str(cfg.kcore_parquet)
    train_path = str(cfg.output_dir / "train_main.csv")
    valid_path = str(cfg.output_dir / "valid_main.csv")
    test_path = str(cfg.output_dir / "test_main.csv")

    # Kiem tra file ton tai
    for p in [parquet_path, train_path, valid_path, test_path]:
        if not os.path.exists(p):
            log.error(f"File not found: {p}. Cannot proceed with integrity check.")
            return

    # Khoi tao DuckDB voi gioi han RAM nghiem ngat
    con = duckdb.connect(':memory:')
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='6GB'")

    # Tao thu muc tam de DuckDB xa rac ra o cung neu can thiet khi dem DISTINCT
    temp_dir = str(cfg.data_dir / "duckdb_integrity_temp")
    os.makedirs(temp_dir, exist_ok=True)
    con.execute(f"PRAGMA temp_directory='{temp_dir}'")

    try:
        # 1. Truy van thong ke tu file Parquet goc (Source of Truth)
        log.info("Step 1/2: Analyzing Original K-Core Parquet...")
        source_query = f"""
            SELECT
                COUNT(*) as total_rows,
                COUNT(DISTINCT user_id) as total_users,
                COUNT(DISTINCT book_id) as total_books
            FROM read_parquet('{parquet_path}')
        """
        src_rows, src_users, src_books = con.execute(source_query).fetchone()

        # 2. Truy van thong ke tu 3 file CSV (UNION ALL giup ghep 3 file thanh 1 luong doc)
        log.info("Step 2/2: Analyzing Exported CSV Files (Train + Valid + Test)...")
        csv_query = f"""
            WITH combined_csv AS (
                SELECT user_id, book_id FROM read_csv_auto('{train_path}')
                UNION ALL
                SELECT user_id, book_id FROM read_csv_auto('{valid_path}')
                UNION ALL
                SELECT user_id, book_id FROM read_csv_auto('{test_path}')
            )
            SELECT
                COUNT(*) as total_rows,
                COUNT(DISTINCT user_id) as total_users,
                COUNT(DISTINCT book_id) as total_books
            FROM combined_csv
        """
        csv_rows, csv_users, csv_books = con.execute(csv_query).fetchone()

        # 3. In bao cao va doi chieu
        log.info("-" * 50)
        log.info("INTEGRITY CHECK REPORT")
        log.info("-" * 50)

        log.info(f"1. Total Interactions (Rows):")
        log.info(f"   - Original Parquet : {src_rows:,}")
        log.info(f"   - Exported CSVs    : {csv_rows:,}")
        log.info(f"   -> Diff            : {src_rows - csv_rows:,}")

        log.info(f"2. Unique Users:")
        log.info(f"   - Original Parquet : {src_users:,}")
        log.info(f"   - Exported CSVs    : {csv_users:,}")
        log.info(f"   -> Diff            : {src_users - csv_users:,}")

        log.info(f"3. Unique Books:")
        log.info(f"   - Original Parquet : {src_books:,}")
        log.info(f"   - Exported CSVs    : {csv_books:,}")
        log.info(f"   -> Diff            : {src_books - csv_books:,}")
        log.info("-" * 50)

        # 4. Ket luan danh gia
        if src_rows == csv_rows and src_users == csv_users and src_books == csv_books:
            log.info("PASSED: The exported datasets perfectly match the original dataset. Zero data loss confirmed.")
        else:
            log.error("FAILED: Discrepancies detected between the original dataset and the exported CSVs.")

    finally:
        # Don dep he thong
        con.close()
        import shutil
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    verify_dataset_integrity(cfg)

2026-04-19 14:00:10 [INFO] IntegrityCheck: Starting Comprehensive Integrity Check...
2026-04-19 14:00:10 [INFO] IntegrityCheck: Step 1/2: Analyzing Original K-Core Parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 14:01:02 [INFO] IntegrityCheck: Step 2/2: Analyzing Exported CSV Files (Train + Valid + Test)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-19 14:03:09 [INFO] IntegrityCheck: --------------------------------------------------
2026-04-19 14:03:09 [INFO] IntegrityCheck: INTEGRITY CHECK REPORT
2026-04-19 14:03:09 [INFO] IntegrityCheck: --------------------------------------------------
2026-04-19 14:03:09 [INFO] IntegrityCheck: 1. Total Interactions (Rows):
2026-04-19 14:03:09 [INFO] IntegrityCheck:    - Original Parquet : 223,587,871
2026-04-19 14:03:09 [INFO] IntegrityCheck:    - Exported CSVs    : 223,587,871
2026-04-19 14:03:09 [INFO] IntegrityCheck:    -> Diff            : 0
2026-04-19 14:03:09 [INFO] IntegrityCheck: 2. Unique Users:
2026-04-19 14:03:09 [INFO] IntegrityCheck:    - Original Parquet : 743,401
2026-04-19 14:03:09 [INFO] IntegrityCheck:    - Exported CSVs    : 743,401
2026-04-19 14:03:09 [INFO] IntegrityCheck:    -> Diff            : 0
2026-04-19 14:03:09 [INFO] IntegrityCheck: 3. Unique Books:
2026-04-19 14:03:09 [INFO] IntegrityCheck:    - Original Parquet : 1,173,713
2026-04-19 14:03:09 [INFO] Int

In [ ]:
# =============================================================================
# Cell 5: Stage 3B - Multi-Target CSV Export (Zero-RAM Streaming)
# =============================================================================

import os
import logging
import polars as pl

log = logging.getLogger("DataSplitPipeline")

def stage_3b_export_csvs(cfg) -> None:
    """
    Exports the ranked dataset into 8 separate CSV files based on temporal
    boundaries and feedback types. Uses Polars streaming (sink_csv) with
    Predicate Pushdown to ensure RAM usage remains near zero.
    """
    if cfg.skip_stage3:
        log.info("Stage 3 skipped via configuration.")
        return

    log.info("Stage 3B: Exporting Data to Final CSV Targets")
    ranked_file = str(cfg.data_dir / "interactions_ranked_tmp.parquet")

    if not os.path.exists(ranked_file):
        raise FileNotFoundError(f"Ranked file not found: {ranked_file}")

    # Create lazy execution graph
    lazy_df = pl.scan_parquet(ranked_file)

    # 1. Define Temporal Logical Conditions
    is_train = pl.col("row_num") <= pl.col("train_end")
    is_valid = (pl.col("row_num") > pl.col("train_end")) & (pl.col("row_num") <= pl.col("valid_end"))
    is_test = pl.col("row_num") > pl.col("valid_end")

    # 2. Define Feedback Logical Conditions
    is_explicit = pl.col("rating").is_between(cfg.explicit_min_rating, cfg.explicit_max_rating)
    is_implicit = pl.col("rating") == 0
    is_pos = is_explicit & (pl.col("rating") >= cfg.positive_min_rating)
    is_neg = is_explicit & (pl.col("rating") <= cfg.negative_max_rating)

    # 3. Map targets to their specific compound conditions
    export_specs = [
        ("train_main", is_train, cfg.outputs["train_main"]),
        ("explicit_train", is_train & is_explicit, cfg.outputs["explicit_train"]),
        ("train_implicit", is_train & is_implicit, cfg.outputs["train_implicit"]),
        ("valid_pos", is_valid & is_pos, cfg.outputs["valid_pos"]),
        ("test_pos", is_test & is_pos, cfg.outputs["test_pos"]),
        ("test_neg", is_test & is_neg, cfg.outputs["test_neg"]),
        ("test_implicit", is_test & is_implicit, cfg.outputs["test_implicit"]),
        ("explicit_test", (is_valid | is_test) & is_explicit, cfg.outputs["explicit_test"]),
    ]

    keep_cols = ["user_id", "book_id", "rating"]

    # 4. Stream data to CSV files
    for name, predicate, out_path in export_specs:
        log.info(f"Streaming {name} to CSV...")
        (
            lazy_df
            .filter(predicate)
            .select(keep_cols)
            .sink_csv(str(out_path))
        )

    log.info("All CSV exports completed successfully.")

    # 5. Cleanup the massive temporary ranked file to free disk space
    if os.path.exists(ranked_file):
        os.remove(ranked_file)
        log.info("Cleaned up temporary ranked parquet file.")


# # =============================================================================
# # Execution Block
# # =============================================================================
# if __name__ == "__main__":
#     try:
#         log.info("Testing Stage 3B Execution...")
#         stage_3b_export_csvs(cfg)
#     except Exception as e:
#         log.error(f"Error during Stage 3B execution: {str(e)}")

In [ ]:
# import logging
# import duckdb
# import pyarrow.parquet as pq
# import gc
# from pathlib import Path

# # Cấu hình log cơ bản
# logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s]: %(message)s")
# log = logging.getLogger("DB_Export")

# def manual_export_from_db(db_path: str, output_parquet_path: str) -> None:
#     log.info(f"Opening existing DuckDB workspace: {db_path}")

#     # Kết nối vào file DB đã lưu
#     con = duckdb.connect(db_path)

#     # Kiểm tra xem bảng interactions có tồn tại không
#     tables = con.execute("SHOW TABLES").fetchall()
#     if ('interactions',) not in tables:
#         log.error("Table 'interactions' not found in the database!")
#         con.close()
#         return

#     total_rows = con.execute("SELECT COUNT(*) FROM interactions").fetchone()[0]
#     log.info(f"Found {total_rows:,} rows ready for export.")

#     log.info("Starting safe export to Parquet...")
#     query_result = con.execute("SELECT * FROM interactions")

#     # Rút từng batch 100.000 dòng
#     record_batch_reader = query_result.fetch_record_batch(rows_per_batch=100_000)

#     writer = None
#     for batch in record_batch_reader:
#         if writer is None:
#             writer = pq.ParquetWriter(output_parquet_path, batch.schema, compression='snappy')
#         writer.write_batch(batch)

#         del batch
#         gc.collect()

#     if writer:
#         writer.close()

#     con.close()
#     log.info(f"Manual export successfully completed! Saved to {output_parquet_path}")

# # Hướng dẫn sử dụng:
# # Thay thế đường dẫn cho đúng với môi trường của bạn
# manual_export_from_db(
#     db_path="/content/drive/My Drive/Thesis/book_recsys/data/kcore_workspace.db",
#     output_parquet_path="/content/drive/My Drive/Thesis/book_recsys/data/interactions_kcore.parquet"
# )

In [ ]:
# =============================================================================
# Cell: Verification (Zero-RAM Out-of-Core Check)
# =============================================================================

import os
import logging
import duckdb

logging.basicConfig(level=logging.INFO,force = True, format="%(asctime)s [%(levelname)s]: %(message)s")
log = logging.getLogger("DataVerification")

def verify_kcore_file_safe(parquet_path: str):
    """
    Verifies the integrity and logic of the K-Core filtered Parquet file
    using DuckDB's out-of-core aggregation to completely avoid RAM spikes.
    """
    if not os.path.exists(parquet_path):
        log.error(f"File not found: {parquet_path}")
        return

    log.info(f"Starting memory-safe verification for: {parquet_path}")

    # Khởi tạo DuckDB và giới hạn RAM cứng ở mức 8GB
    con = duckdb.connect(database=':memory:')
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='8GB'")

    try:
        # 1. Kiểm tra tổng số dòng
        log.info("Counting total rows...")
        total_rows = con.execute(f"SELECT COUNT(*) FROM '{parquet_path}'").fetchone()[0]

        # 2. Tìm số tương tác ít nhất của một User
        log.info("Calculating minimum interactions per user...")
        min_user_interactions = con.execute(f"""
            SELECT MIN(interact_count) FROM (
                SELECT COUNT(*) as interact_count
                FROM '{parquet_path}'
                WHERE rating > 3
                GROUP BY user_id
            )
        """).fetchone()[0]
        max_user_interaction = con.execute(f"""
          SELECT MAX(interact_count) FROM (
                SELECT COUNT(*) as interact_count
                FROM '{parquet_path}'
                GROUP BY user_id
            )
        """).fetchone()[0]

        # 3. Tìm số tương tác ít nhất của một Sách
        log.info("Calculating minimum interactions per book...")
        min_book_interactions = con.execute(f"""
            SELECT MIN(interact_count) FROM (
                SELECT COUNT(*) as interact_count
                FROM '{parquet_path}'
                GROUP BY book_id
            )
        """).fetchone()[0]

        # In Báo Cáo
        log.info("-" * 40)
        log.info("VERIFICATION REPORT")
        log.info("-" * 40)
        log.info(f"Total Rows: {total_rows:,}")
        log.info(f"Min interactions per User: {min_user_interactions}")
        log.info(f"Min interactions per Book: {min_book_interactions}")
        log.info(f"Max interactions per User: {max_user_interaction}")
        log.info("-" * 40)

        # Đánh giá logic
        if min_user_interactions >= cfg.k_user and min_book_interactions >= cfg.k_book:
            log.info("SUCCESS: K-Core constraints are perfectly met!")
        else:
            log.warning("WARNING: Constraints failed. Sparse data still exists.")

    except Exception as e:
        log.error(f"Verification failed: {e}")
    finally:
        con.close()

# # Thực thi (Chỉ cần truyền đúng biến đường dẫn hoặc cfg.kcore_parquet)
# if __name__ == "__main__":
#     # Giả định biến cfg đã tồn tại từ Cell 1, nếu chạy riêng thì thay bằng string đường dẫn:
#     # verify_kcore_file_safe("/content/drive/My Drive/Thesis/book_recsys/data/interactions_kcore.parquet")

#     input_file = str(cfg.kcore_parquet)
#     explicit_out = str(cfg.output_dir / "interactions_explicit.parquet")
#     implicit_out = str(cfg.output_dir / "interactions_implicit.parquet")
#     verify_kcore_file_safe(str(cfg.kcore_parquet))
#     verify_kcore_file_safe(explicit_out)
#     verify_kcore_file_safe(implicit_out)

2026-04-18 10:24:36,467 [INFO]: Starting memory-safe verification for: /content/drive/My Drive/Thesis/book_recsys/data/interactions_kcore.parquet
2026-04-18 10:24:36,683 [INFO]: Counting total rows...
2026-04-18 10:24:36,762 [INFO]: Calculating minimum interactions per user...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 10:24:56,759 [INFO]: Calculating minimum interactions per book...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

2026-04-18 10:26:31,309 [INFO]: ----------------------------------------
2026-04-18 10:26:31,311 [INFO]: VERIFICATION REPORT
2026-04-18 10:26:31,313 [INFO]: ----------------------------------------
2026-04-18 10:26:31,314 [INFO]: Total Rows: 101,656,118
2026-04-18 10:26:31,315 [INFO]: Min interactions per User: 1
2026-04-18 10:26:31,317 [INFO]: Min interactions per Book: 1
2026-04-18 10:26:31,318 [INFO]: Max interactions per User: 36191
2026-04-18 10:26:31,319 [INFO]: ----------------------------------------
2026-04-18 10:26:31,321 [WARNING]: WARNING: Constraints failed. Sparse data still exists.
2026-04-18 10:26:31,349 [INFO]: Starting memory-safe verification for: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/interactions_implicit.parquet
2026-04-18 10:26:31,367 [INFO]: Counting total rows...
2026-04-18 10:26:32,969 [INFO]: Calculating minimum interactions per user...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 10:27:10,430 [INFO]: Calculating minimum interactions per book...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 10:27:36,296 [INFO]: ----------------------------------------
2026-04-18 10:27:36,298 [INFO]: VERIFICATION REPORT
2026-04-18 10:27:36,298 [INFO]: ----------------------------------------
2026-04-18 10:27:36,299 [INFO]: Total Rows: 121,931,753
2026-04-18 10:27:36,300 [INFO]: Min interactions per User: None
2026-04-18 10:27:36,301 [INFO]: Min interactions per Book: 1
2026-04-18 10:27:36,302 [INFO]: Max interactions per User: 113199
2026-04-18 10:27:36,303 [INFO]: ----------------------------------------
2026-04-18 10:27:36,305 [ERROR]: Verification failed: '>=' not supported between instances of 'NoneType' and 'int'
